In [ ]:
# Get data and images
import requests
from tqdm import tqdm
import os

for id in tqdm([flat["propId"] for flat in desired_flats if flat["propId"] not in data["flats"]]):
    data["flats"][id] = requests.get(f'https://cityexpert.rs/api/PropertyView/{id}/r').json()

image_names = [url for flat in data["flats"].values() for url in flat.get("onsite", {}).get("imgFiles", [])]
images = [("https://img.cityexpert.rs/sites/default/files/styles/1280x/public/image/" + name, "./imgs/" + name) for name in set(image_names)]

for url, path in tqdm([src for src in images if not os.path.exists(src[1])]):
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()

        with open(path, 'wb') as f:
            f.write(response.content)
    except Exception as e:
        print(e)

save()

In [ ]:
from openai import OpenAI

client = OpenAI()

prompt = "This is photos of apartment in renting websites provided by landlord. They may be photoshopped and"
urls = []

response = client.chat.completions.create(
  model="gpt-4-1106-vision-preview",
  max_tokens=4096,
  messages=[{
      "role": "user",
      "content": [{"type": "text", "text": prompt}] + \
        [{"type": "image_url", "image_url": {"url": url}} for url in urls]
    }]
)

response.choices[0].message.content

In [ ]:
from IPython.display import display
from PIL import Image

paths = ["./imgs/" + path for path in data["flats"]["36159"]["onsite"]["imgFiles"]]

size = 400

for path in paths:
    img = Image.open(path)
    img = img.resize((size, int(size * img.height / img.width)))
    display(img)


In [ ]:
flat = {key: value for key, value in data["flats"]["36159"].items() if key not in ["neighbourhoods"]}

print(json.dumps(flat, ensure_ascii=False, indent=2))